# TEST IPSA - Estrategia Híbrida
## Yahoo Finance (close/1D) + Banco Central Chile (MTD/YTD)

In [2]:
# Imports
import os
import requests
import yfinance as yf
import json
from datetime import date, datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

ModuleNotFoundError: No module named 'dotenv'

## PASO 1: Yahoo Finance - Close y 1D

In [3]:
# Obtener datos de Yahoo Finance
ticker = yf.Ticker("^IPSA")
hist = ticker.history(period="5d")

print(f"Datos recibidos: {len(hist)} días")
print("\nÚltimos 5 días:")
print(hist[['Close']])

# Extraer valores
yahoo_close = float(hist['Close'].iloc[-1])
yahoo_date = hist.index[-1].strftime('%Y-%m-%d')

if len(hist) >= 2:
    prev_close = float(hist['Close'].iloc[-2])
    yahoo_change_1d = ((yahoo_close / prev_close) - 1.0) * 100.0
else:
    yahoo_change_1d = None

print(f"\n✓ Close: {yahoo_close:,.2f}")
print(f"✓ Fecha: {yahoo_date}")
print(f"✓ Change 1D: {yahoo_change_1d:+.2f}%" if yahoo_change_1d else "✓ Change 1D: N/A")

Datos recibidos: 1 días

Últimos 5 días:
                                  Close
Date                                   
2025-12-19 00:00:00-03:00  10304.259766

✓ Close: 10,304.26
✓ Fecha: 2025-12-19
✓ Change 1D: N/A


## PASO 2: Banco Central Chile - Niveles Históricos

In [4]:
# Configuración BCCh
BCCH_BDE_URL = "https://si3.bcentral.cl/SieteRestWS/SieteRestWS.ashx"
IPSA_SERIES = "F013.IBC.IND.N.7.LAC.CL.CLP.BLO.D"

bcch_user = os.getenv("BCCH_USER")
bcch_pass = os.getenv("BCCH_PASS")

if not bcch_user or not bcch_pass:
    print("❌ ERROR: Configura BCCH_USER y BCCH_PASS en .env")
else:
    print(f"✓ Credenciales: {bcch_user[:3]}***")

✓ Credenciales: pab***


In [5]:
# Request a BCCh
end_date = date.today()
start_date = end_date - timedelta(days=400)

params = {
    "user": bcch_user,
    "pass": bcch_pass,
    "firstdate": start_date.strftime("%Y-%m-%d"),
    "lastdate": end_date.strftime("%Y-%m-%d"),
    "timeseries": IPSA_SERIES,
    "function": "GetSeries",
}

print(f"Consultando BCCh desde {start_date} hasta {end_date}...")

response = requests.get(BCCH_BDE_URL, params=params, timeout=30)
data = response.json()

print(f"Status: {response.status_code}")

Consultando BCCh desde 2024-11-15 hasta 2025-12-20...
Status: 200


In [9]:
# Parsear observaciones
series_obj = data.get("Series", {}).get("Obs", [])

observations = []
for obs in series_obj:
    val_str = obs.get("value", "").strip()
    if val_str:
        try:
            observations.append({
                "value": float(val_str),
                "date": obs.get("indexDateString")
            })
        except:
            pass

print(f"✓ Observaciones válidas: {len(observations)}")
print("\nÚltimas 10 observaciones:")
for obs in observations[-10:]:
    print(f"  {obs['date']}: {obs['value']:,.2f}")

✓ Observaciones válidas: 399

Últimas 10 observaciones:
  09-12-2025: 10,179.97
  10-12-2025: 10,170.37
  11-12-2025: 10,362.86
  12-12-2025: 10,400.01
  13-12-2025: nan
  14-12-2025: nan
  15-12-2025: 10,302.23
  16-12-2025: 10,188.43
  17-12-2025: 10,134.52
  18-12-2025: 10,194.26


## PASO 3: Calcular MTD y YTD

In [10]:
observations

[{'value': 6527.02, 'date': '15-11-2024'},
 {'value': nan, 'date': '16-11-2024'},
 {'value': nan, 'date': '17-11-2024'},
 {'value': 6542.39, 'date': '18-11-2024'},
 {'value': 6549.37, 'date': '19-11-2024'},
 {'value': 6576.33, 'date': '20-11-2024'},
 {'value': 6592.94, 'date': '21-11-2024'},
 {'value': 6564.16, 'date': '22-11-2024'},
 {'value': nan, 'date': '23-11-2024'},
 {'value': nan, 'date': '24-11-2024'},
 {'value': 6549.16, 'date': '25-11-2024'},
 {'value': 6558.3, 'date': '26-11-2024'},
 {'value': 6578.98, 'date': '27-11-2024'},
 {'value': 6587.35, 'date': '28-11-2024'},
 {'value': 6576.6, 'date': '29-11-2024'},
 {'value': nan, 'date': '30-11-2024'},
 {'value': nan, 'date': '01-12-2024'},
 {'value': 6639.86, 'date': '02-12-2024'},
 {'value': 6629.72, 'date': '03-12-2024'},
 {'value': 6631.36, 'date': '04-12-2024'},
 {'value': 6660.14, 'date': '05-12-2024'},
 {'value': 6648.65, 'date': '06-12-2024'},
 {'value': nan, 'date': '07-12-2024'},
 {'value': nan, 'date': '08-12-2024'},
 {

In [7]:
# MTD - Mes corrido
today = date.today()
first_day_month = today.replace(day=1)

print(f"Buscando nivel desde: {first_day_month}")

month_start_value = None
month_start_date = None

for obs in observations:
    try:
        obs_date = datetime.strptime(obs["date"], "%Y-%m-%d").date()
        if obs_date >= first_day_month:
            month_start_value = obs["value"]
            month_start_date = obs["date"]
            break
    except:
        pass

if month_start_value:
    mtd = ((yahoo_close / month_start_value) - 1.0) * 100.0
    print(f"✓ Nivel inicio mes ({month_start_date}): {month_start_value:,.2f}")
    print(f"✓ MTD = ({yahoo_close:,.2f} / {month_start_value:,.2f} - 1) × 100")
    print(f"✓ MTD = {mtd:+.2f}%")
else:
    mtd = None
    print("❌ No se encontró nivel de inicio de mes")

Buscando nivel desde: 2025-12-01
❌ No se encontró nivel de inicio de mes


In [8]:
# YTD - Año corrido
first_day_year = today.replace(month=1, day=1)

print(f"Buscando nivel desde: {first_day_year}")

year_start_value = None
year_start_date = None

for obs in observations:
    try:
        obs_date = datetime.strptime(obs["date"], "%Y-%m-%d").date()
        if obs_date >= first_day_year:
            year_start_value = obs["value"]
            year_start_date = obs["date"]
            break
    except:
        pass

if year_start_value:
    ytd = ((yahoo_close / year_start_value) - 1.0) * 100.0
    print(f"✓ Nivel inicio año ({year_start_date}): {year_start_value:,.2f}")
    print(f"✓ YTD = ({yahoo_close:,.2f} / {year_start_value:,.2f} - 1) × 100")
    print(f"✓ YTD = {ytd:+.2f}%")
else:
    ytd = None
    print("❌ No se encontró nivel de inicio de año")

Buscando nivel desde: 2025-01-01
❌ No se encontró nivel de inicio de año


## RESULTADO FINAL

In [ ]:
# Consolidar resultado
result = {
    "close": round(yahoo_close, 2),
    "date": yahoo_date,
    "change_1d": round(yahoo_change_1d, 2) if yahoo_change_1d else None,
    "change_mtd": round(mtd, 2) if mtd is not None else None,
    "change_ytd": round(ytd, 2) if ytd is not None else None,
    "source_close": "Yahoo Finance",
    "source_historical": "Banco Central de Chile"
}

print("="*60)
print("IPSA - RESULTADO HÍBRIDO")
print("="*60)
print(json.dumps(result, indent=2, ensure_ascii=False))
print()
print(f"Close:      {result['close']:,.2f}  (Yahoo Finance)")
print(f"Fecha:      {result['date']}")
print(f"Change 1D:  {result['change_1d']:+.2f}%  (Yahoo Finance)" if result['change_1d'] else "Change 1D:  N/A")
print(f"Change MTD: {result['change_mtd']:+.2f}%  (BCCh)" if result['change_mtd'] else "Change MTD: N/A")
print(f"Change YTD: {result['change_ytd']:+.2f}%  (BCCh)" if result['change_ytd'] else "Change YTD: N/A")